# PyMembrane: first simulation notebook

<img src="../_static/logo-github.png" alt="PyMembrane logo" width="220">

This notebook is a beginner-friendly, step-by-step tutorial for running a very short PyMembrane simulation.

It reuses the **existing periodic sheet example physics** that is already packaged with PyMembrane. The goal here is not a production simulation. It is a lightweight tutorial and smoke test that shows the standard workflow:

1. import the core classes,
2. locate bundled example input files,
3. create a simulation box,
4. create a `System`,
5. read a mesh,
6. add forces,
7. add an integrator,
8. run a short simulation,
9. write output files.


## Installation reminder

Install PyMembrane from the repository root:

```bash
pip install -e .
python -c "import pymembrane; print(pymembrane.__file__)"
```

The same packaged example physics can also be run directly without opening this notebook:

```bash
python -m pymembrane.examples.periodic --quick
```

Jupyter itself is optional. PyMembrane does not require Jupyter for installation.


In [ ]:
from math import sqrt
from pathlib import Path

from pymembrane import Box, System, Evolver
from pymembrane.examples._resources import example_data_path


## 1. Locate bundled input data

The packaged examples include their own mesh files. This means that new users do **not** need to download extra files manually.

The helper below resolves the packaged data in the same way as the installed example modules.


In [ ]:
with example_data_path("02_periodic/vertices.dat") as vertex_file, example_data_path("02_periodic/faces.dat") as face_file:
    print("vertices:", vertex_file)
    print("faces   :", face_file)


## 2. Create the simulation box

This tutorial uses the existing periodic-sheet example. The box is periodic and matches the packaged `periodic` example.


In [ ]:
box = Box(sqrt(3.0) * 29, 50.0, 50.0, True, True, True)
print(box)


## 3. Create the System and read the mesh

The `System` stores the mesh and boundary conditions. The mesh files are read from the bundled example data.


In [ ]:
system = System(box)

with example_data_path("02_periodic/vertices.dat") as vertex_file, example_data_path("02_periodic/faces.dat") as face_file:
    system.read_mesh_from_files(
        files={"vertices": str(vertex_file), "faces": str(face_file)}
    )

system.enforce_boundaries()

print(f"vertices: {system.Numvertices}")
print(f"faces   : {system.Numfaces}")
print(f"edges   : {system.Numedges}")


## 4. Create the Evolver and add forces

This tutorial keeps the same model ingredients as the packaged periodic example:

- harmonic stretching,
- edge-length limiting,
- dihedral bending.

These are the same public API calls you would use in your own scripts.


In [ ]:
evolver = Evolver(system)
evolver.add_force("Mesh>Harmonic", {"k": {"0": "100.0"}, "l0": {"0": "1.0"}})
evolver.add_force("Mesh>Limit", {"lmin": {"0": "0.7"}, "lmax": {"0": "1.3"}})
evolver.add_force("Mesh>Bending>Dihedral", {"kappa": {"0": "1.0"}})


## 5. Add an integrator and set the temperature

The periodic example uses Brownian dynamics vertex moves. For a tutorial run we keep the same integrator type and parameters, but run only a very small number of steps.


In [ ]:
evolver.add_integrator("Mesh>Brownian>vertex>move", {"seed": "202208"})
evolver.set_time_step("2e-3")
evolver.set_global_temperature("1e-4")


## 6. Run a short tutorial simulation

This is intentionally a smoke/tutorial run. It is long enough to exercise the workflow, but short enough to remain lightweight for reviewers.


In [ ]:
tutorial_output = Path("tutorial_output")
tutorial_output.mkdir(exist_ok=True)

initial_energy = system.compute.energy(evolver)
print("initial energy:", initial_energy)

evolver.evolveMD(steps=10)

final_energy = system.compute.energy(evolver)
print("final energy:", final_energy)


## 7. Write output files

PyMembrane's default dumper can write legacy ASCII VTK directly from Python. This does **not** require the VTK Python bindings.

We also write an OBJ file for lightweight geometry inspection.


In [ ]:
system.dumper.vtk(str(tutorial_output / "tutorial_output"), periodic=True)
system.dumper.obj(str(tutorial_output / "tutorial_output"))

for path in sorted(tutorial_output.iterdir()):
    print(path.name)


## Expected output

After the notebook runs, you should see files such as:

- `tutorial_output/tutorial_output.vtk`
- `tutorial_output/tutorial_output.obj`

Open the `.vtk` file in **ParaView** if you want to inspect the mesh geometry. The `.obj` file is useful for quick geometry export to general 3D tools.


## Where to go next

- Packaged examples: `python -m pymembrane.examples.periodic --quick`
- Hybrid example: `python -m pymembrane.examples.hybrid_mc_bd --quick`
- Documentation examples: `docs/examples/`
- Python API reference: `docs/pythonapi/`

For larger or publication-scale runs, use the packaged examples as the canonical physics entry points rather than this tutorial smoke run.
